<a href="https://colab.research.google.com/github/AaronL123/Flyrank-ML-assignments/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AaronL123/Flyrank-ML-assignments/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

## Two signal checks, before the rule

**Signal 1 — staleness (behind FlyRank's refresh flags): FALSE.**

The refresh flags lean on "how long since this page was last updated." In this warehouse that would come from `last_optimized_date`. It does not survive checking:

- 50.6% of my 59,122 pages have no value at all.
- Of the 29,194 that do, **100% fall after my decision moment** — between 34 and 107 days later (median 82).

So `last_optimized_date` is not a record of when a page was last refreshed; it is forward-looking — a scheduled or planned optimization date. Using it as a staleness feature would pull information from after the decision into the decision itself. **My rule cannot use staleness, and the check is what caught it.** A rule built on the obvious refresh signal would have leaked without ever looking wrong.

**Signal 2 — position tier vs decline (behind the CTR-fix logic): CONFIRMED.**

| Position tier | n | decline rate |
|---|---|---|
| 1–3 | 7,344 | 61.2% |
| 4–10 | 32,068 | 63.4% |
| 11–20 | 10,551 | 70.6% |
| 21–50 | 8,694 | 71.8% |
| 50+ | 445 | 89.7% |

Decline rate rises monotonically with worse position, from 61.2% to 89.7%, against an overall base rate of 65.9%. The relationship is directionally consistent and every bucket has usable n. Median CTR also drops in the 21–50 tier (0.23% against ~0.38% higher up), which is what the CTR-fix logic assumes.

**Caution:** the spread is real but modest across the tiers that hold most of the data — 61.2% to 71.8% covers 58,657 of 59,122 pages. The 89.7% tier is only 445 pages. So position is a usable signal, not a strong one, and a rule leaning on it alone will not separate much.

## The rule

Plain words: **a page is worth reviewing first if it already has real search demand, is ranking outside the top positions, and is converting that demand into clicks poorly.** Demand means someone is looking; weak position and weak CTR mean the page is present but not capturing it.

Reason codes (one per page, first match wins):

- `weak_position_high_demand` — ranks outside the top 10 with substantial impressions
- `low_ctr_good_position` — ranks top 10 but CTR is below the tier median
- `thin_engagement` — has impressions but very few clicks relative to volume

Action labels: `refresh`, `optimize_title_meta`, `review_manually`.

Staleness is deliberately absent — signal 1 ruled it out.

In [2]:
!pip install -q duckdb

import duckdb, pandas as pd, numpy as np, json, os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN").strip()
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL       = "hf://datasets/FlyRank/internship-warehouse"
CONTENT   = f"{REL}/dim_content.parquet"
DEV_MONTH = f"{REL}/fact_content_daily_performance/month=2026-03/data_0.parquet"
FEAT_END, OUT_START = "2026-03-21", "2026-03-22"

frame = con.sql(f"""
    WITH feat AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions)            AS impressions_21d,
               SUM(gsc_clicks)                 AS clicks_21d,
               AVG(NULLIF(gsc_avg_position,0)) AS avg_position_21d,
               COUNT(*)                        AS days_observed
        FROM read_parquet('{DEV_MONTH}')
        WHERE gsc_data_available IS TRUE AND report_date <= DATE '{FEAT_END}'
        GROUP BY 1,2
        HAVING SUM(gsc_clicks) > 0          -- label can only fire where clicks exist
    ),
    outcome AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_clicks) AS clicks_out, COUNT(*) AS days_out
        FROM read_parquet('{DEV_MONTH}')
        WHERE gsc_data_available IS TRUE AND report_date >= DATE '{OUT_START}'
        GROUP BY 1,2
    )
    SELECT f.*, o.clicks_out, o.days_out,
           c.last_optimized_date, c.content_created_date,
           c.content_type, c.search_volume, c.word_count
    FROM feat f
    JOIN outcome o USING (client_hash_id, content_hash_id)
    LEFT JOIN read_parquet('{CONTENT}') c USING (client_hash_id, content_hash_id)
    WHERE o.days_out > 0
""").df()

frame["ctr_21d"]      = frame.clicks_21d / frame.impressions_21d
frame["rate_before"]  = frame.clicks_21d / frame.days_observed
frame["rate_after"]   = frame.clicks_out / frame.days_out
frame["is_declining"] = (frame.rate_after < frame.rate_before).astype(int)
frame["days_since_update"] = (pd.Timestamp("2026-03-21") - pd.to_datetime(frame.last_optimized_date)).dt.days

print(f"Pages: {len(frame):,}   Base rate: {frame.is_declining.mean():.1%}")
print(f"Missing last_optimized_date: {frame.last_optimized_date.isna().mean():.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages: 59,122   Base rate: 65.9%
Missing last_optimized_date: 50.6%


In [4]:
# --- Signal 1: staleness (behind FlyRank's refresh flags) ---
s = frame.dropna(subset=["days_since_update"]).copy()
s["bucket"] = pd.cut(s.days_since_update,
                     bins=[-1, 30, 90, 180, 365, 10**6],
                     labels=["0-30d", "31-90d", "91-180d", "181-365d", "365d+"])
t1 = s.groupby("bucket", observed=True).agg(n=("is_declining", "size"),
                                            decline_rate=("is_declining", "mean"))
t1["decline_rate"] = (100 * t1.decline_rate).round(1)
print("SIGNAL 1 — staleness vs decline")
print(f"(usable rows: {len(s):,} of {len(frame):,}; {frame.days_since_update.isna().mean():.1%} have no update date)")
print(t1.to_string())
print(f"overall base rate: {100*frame.is_declining.mean():.1f}%\n")

# --- Signal 2: CTR vs position (behind the CTR-fix logic) ---
p = frame.dropna(subset=["avg_position_21d"]).copy()
p["pos_tier"] = pd.cut(p.avg_position_21d,
                       bins=[0, 3, 10, 20, 50, 10**6],
                       labels=["1-3", "4-10", "11-20", "21-50", "50+"])
t2 = p.groupby("pos_tier", observed=True).agg(n=("is_declining", "size"),
                                              decline_rate=("is_declining", "mean"),
                                              median_ctr=("ctr_21d", "median"))
t2["decline_rate"] = (100 * t2.decline_rate).round(1)
t2["median_ctr"]   = (100 * t2.median_ctr).round(2)
print("SIGNAL 2 — position tier vs decline and CTR")
print(t2.to_string())

SIGNAL 1 — staleness vs decline
(usable rows: 29,194 of 59,122; 50.6% have no update date)
Empty DataFrame
Columns: [n, decline_rate]
Index: []
overall base rate: 65.9%

SIGNAL 2 — position tier vs decline and CTR
              n  decline_rate  median_ctr
pos_tier                                 
1-3        7344          61.2        0.37
4-10      32068          63.4        0.38
11-20     10551          70.6        0.38
21-50      8694          71.8        0.23
50+         445          89.7        0.79
SIGNAL 1 — staleness vs decline
(usable rows: 29,194 of 59,122; 50.6% have no update date)
Empty DataFrame
Columns: [n, decline_rate]
Index: []
overall base rate: 65.9%

SIGNAL 2 — position tier vs decline and CTR
              n  decline_rate  median_ctr
pos_tier                                 
1-3        7344          61.2        0.37
4-10      32068          63.4        0.38
11-20     10551          70.6        0.38
21-50      8694          71.8        0.23
50+         445          8

In [5]:
d = frame.days_since_update.dropna()
print(f"n = {len(d):,}")
print(d.describe())
print(f"\nNegative (optimized after 2026-03-21): {(d < 0).sum():,}  ({(d < 0).mean():.1%})")
print(f"Zero or positive: {(d >= 0).sum():,}")

n = 29,194
count    29194.000000
mean       -81.609646
std         15.827310
min       -107.000000
25%        -95.000000
50%        -82.000000
75%        -66.000000
max        -34.000000
Name: days_since_update, dtype: float64

Negative (optimized after 2026-03-21): 29,194  (100.0%)
Zero or positive: 0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.